# Итоговое соревнование по модулю. Классификация ансамблями: риск ухода пользователя

Это итоговая задача в формате учебного leaderboard после деревьев, случайного леса и градиентного бустинга.

**Задача:** предсказать `churn` — уйдёт ли пользователь из образовательного сервиса в ближайший месяц.

**Главная метрика соревнования:** ROC-AUC. Больше — лучше.

Интуитивно ROC-AUC проверяет, насколько хорошо модель ставит более высокий риск тем пользователям, которые действительно ушли.


## Правила и ориентир на 90 минут

1. 0–10 мин — понять данные, метрику и baseline.
2. 10–30 мин — обучить первый ансамбль.
3. 30–65 мин — сравнить несколько ансамблей.
4. 65–80 мин — улучшить признаки и параметры.
5. 80–90 мин — сохранить submission и обсудить победившие идеи.

Обязательный набор моделей для сравнения:

- `RandomForestClassifier`;
- `ExtraTreesClassifier`;
- `GradientBoostingClassifier`;
- `HistGradientBoostingClassifier`;
- `XGBClassifier`;
- `LGBMClassifier`;
- `CatBoostClassifier`.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import (
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

RANDOM_STATE = 42
DATA_DIR = Path("data")


In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

print(train.shape, test.shape)
display(train.head())
display(sample_submission.head())


## Validation

Доля класса 1 может быть не 50/50, поэтому используем `stratify=y`.


In [ ]:
target = "churn"
id_col = "id"

features = [c for c in train.columns if c not in [target, id_col]]
X = train[features]
y = train[target]

print("Доля ушедших пользователей:", round(y.mean(), 3))

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)


In [ ]:
def make_ohe():
    # Для HistGradientBoosting удобнее получить плотную матрицу, а не sparse.
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

numeric_preprocess = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

categorical_preprocess = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_ohe()),
    ]
)

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_preprocess, numeric_features),
        ("cat", categorical_preprocess, categorical_features),
    ]
)


## Baseline и функция оценки

Baseline здесь почти ничего не знает об объектах. Хорошая модель должна заметно обогнать его.


In [ ]:
def evaluate_model(name, model, X_train, y_train, X_val, y_val):
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_val)[:, 1]
    pred = (proba >= 0.5).astype(int)
    return {
        "model": name,
        "roc_auc": roc_auc_score(y_val, proba),
        "average_precision": average_precision_score(y_val, proba),
        "f1_at_0_5": f1_score(y_val, pred, zero_division=0),
        "precision_at_0_5": precision_score(y_val, pred, zero_division=0),
        "recall_at_0_5": recall_score(y_val, pred, zero_division=0),
    }


dummy = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("model", DummyClassifier(strategy="prior")),
    ]
)

pd.DataFrame([evaluate_model("Dummy prior", dummy, X_train, y_train, X_val, y_val)])


## Сравнение ансамблей

Сначала сравните несколько моделей “как есть”, а потом улучшайте самую перспективную.


In [ ]:
models = {
    "RandomForest": RandomForestClassifier(
        n_estimators=250,
        max_depth=None,
        min_samples_leaf=3,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "ExtraTrees": ExtraTreesClassifier(
        n_estimators=250,
        max_depth=None,
        min_samples_leaf=3,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "GradientBoosting": GradientBoostingClassifier(
        n_estimators=160,
        learning_rate=0.05,
        max_depth=3,
        random_state=RANDOM_STATE,
    ),
    "HistGradientBoosting": HistGradientBoostingClassifier(
        max_iter=180,
        learning_rate=0.05,
        max_leaf_nodes=31,
        random_state=RANDOM_STATE,
    ),
    "XGBoost": XGBClassifier(
        n_estimators=220,
        learning_rate=0.05,
        max_depth=3,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=220,
        learning_rate=0.05,
        num_leaves=31,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1,
    ),
    "CatBoost": CatBoostClassifier(
        iterations=220,
        learning_rate=0.05,
        depth=4,
        loss_function="Logloss",
        random_seed=RANDOM_STATE,
        verbose=False,
    ),
}

rows = []
trained_models = {}

for name, estimator in models.items():
    pipe = Pipeline(
        steps=[
            ("preprocess", preprocess),
            ("model", estimator),
        ]
    )
    rows.append(evaluate_model(name, pipe, X_train, y_train, X_val, y_val))
    trained_models[name] = pipe

leaderboard = pd.DataFrame(rows).sort_values("roc_auc", ascending=False)
leaderboard


## Идеи для улучшения

Что можно пробовать:

- `n_estimators`: больше деревьев часто лучше, но медленнее;
- `max_depth` или `max_leaf_nodes`: ограничивают сложность деревьев;
- `min_samples_leaf`: делает деревья спокойнее и иногда улучшает качество;
- `learning_rate` и `n_estimators` для бустинга;
- признаки вроде `sessions_per_age_day`, `price_after_discount`, `is_inactive`;
- усреднение вероятностей двух сильных моделей.

Главное правило: сравнивайте варианты по одной и той же validation-выборке.


In [ ]:
def add_features(df):
    df = df.copy()
    df["sessions_per_100_days"] = df["sessions_last_30"] / (df["account_age_days"] / 100 + 1)
    df["price_after_discount"] = df["plan_price"] * (1 - df["discount_percent"] / 100)
    df["is_inactive"] = (df["days_since_last_activity"] >= 14).astype(int)
    df["support_per_session"] = df["support_tickets"] / (df["sessions_last_30"] + 1)
    return df


train_fe = add_features(train)
test_fe = add_features(test)

features_fe = [c for c in train_fe.columns if c not in [target, id_col]]
X = train_fe[features_fe]
y = train_fe[target]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

preprocess_fe = ColumnTransformer(
    transformers=[
        ("num", numeric_preprocess, numeric_features),
        ("cat", categorical_preprocess, categorical_features),
    ]
)

best_model = Pipeline(
    steps=[
        ("preprocess", preprocess_fe),
        (
            "model",
            GradientBoostingClassifier(
                n_estimators=220,
                learning_rate=0.04,
                max_depth=3,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

report = evaluate_model("Tuned GradientBoosting", best_model, X_train, y_train, X_val, y_val)
pd.DataFrame([report])


## Финальное обучение и submission

Для ROC-AUC нужно сдавать вероятность класса 1, а не только 0/1.


In [ ]:
final_X = train_fe[features_fe]
final_y = train_fe[target]

final_model = best_model
final_model.fit(final_X, final_y)

test_proba = final_model.predict_proba(test_fe[features_fe])[:, 1]

submission = pd.DataFrame(
    {
        "id": test_fe["id"],
        "churn_probability": np.round(test_proba, 5),
    }
)

submission.to_csv("submission.csv", index=False)
submission.head()


## Что сдать

Сдайте файл `submission.csv`.

В нём должны быть ровно два столбца:

- `id`;
- `churn_probability` — вероятность ухода пользователя.

Если работаете в команде, переименуйте файл в понятное имя, например `team_ivan_masha.csv`.
